In [ ]:
# Task 1: Vehicle_ID Trimming & Canonical Case
import pandas as pd

df = pd.read_csv("../Datasets/car_rental_dirty_dataset_new1.csv")


1. Vehicle_ID trimming and canonical case.

In [ ]:
unique_vid = df['Vehicle_ID'].nunique()

# Inspect raw Vehicle_ID values
print(f"Shape: {df.shape}")
print(f"Unique Vehicle_IDs (raw): {unique_vid}")
print(sorted(df['Vehicle_ID'].dropna().unique()))
print(f"\n{df['Vehicle_ID'].value_counts().sort_index()}")

Shape: (5000, 21)
Unique Vehicle_IDs (raw): 601
[' CAR-010 ', ' CAR-014 ', ' CAR-015 ', ' CAR-027 ', ' CAR-034 ', ' CAR-045 ', ' CAR-047 ', ' CAR-063 ', ' CAR-066 ', ' CAR-070 ', ' CAR-089 ', ' CAR-091 ', ' CAR-100 ', ' CAR-101 ', ' CAR-104 ', ' CAR-112 ', ' CAR-128 ', ' CAR-133 ', ' CAR-135 ', ' CAR-137 ', ' CAR-138 ', ' CAR-148 ', ' CAR-164 ', ' CAR-169 ', ' CAR-188 ', ' CAR-190 ', ' CAR-193 ', ' CAR-195 ', ' CAR-206 ', ' CAR-213 ', ' CAR-263 ', ' CAR-275 ', ' CAR-282 ', ' CAR-286 ', ' CAR-301 ', ' CAR-303 ', ' CAR-317 ', ' CAR-333 ', ' CAR-340 ', ' CAR-343 ', ' CAR-347 ', ' CAR-369 ', ' CAR-374 ', ' CAR-379 ', ' CAR-381 ', ' CAR-385 ', ' CAR-388 ', ' CAR-396 ', ' CAR-402 ', ' CAR-430 ', ' CAR-432 ', ' CAR-435 ', ' CAR-436 ', ' CAR-440 ', ' CAR-453 ', ' CAR-471 ', ' CAR-477 ', ' CAR-480 ', ' CAR-503 ', ' CAR-504 ', ' CAR-520 ', ' CAR-533 ', ' CAR-538 ', ' CAR-564 ', ' CAR-572 ', ' CAR-583 ', ' CAR-591 ', ' CAR-594 ', ' CAR-599 ', 'CAR 005', 'CAR 011', 'CAR 020', 'CAR 030', 'CAR 031',

In [ ]:
# Clean Vehicle_ID: trim whitespace, normalize separators, canonical uppercase

# Strip leading/trailing whitespace
df['Vehicle_ID'] = df['Vehicle_ID'].str.strip()

# Normalize separators: double hyphens -> single, underscores -> hyphens
df['Vehicle_ID'] = df['Vehicle_ID'].str.replace('--', '-', regex=False)
df['Vehicle_ID'] = df['Vehicle_ID'].str.replace('_', '-', regex=False)

# Insert hyphen between letters and digits where missing (e.g., CAR08 -> CAR-08)
df['Vehicle_ID'] = df['Vehicle_ID'].str.replace(r'([A-Za-z])\s+(\d)', r'\1-\2', regex=True)
df['Vehicle_ID'] = df['Vehicle_ID'].str.replace(r'([A-Za-z])(\d)', r'\1-\2', regex=True)

# Convert to uppercase canonical form
df['Vehicle_ID'] = df['Vehicle_ID'].str.upper()

# Fix CAR-004 -> CAR-04 to match 2-digit format
df['Vehicle_ID'] = df['Vehicle_ID'].replace('CAR-004', 'CAR-04')

# Convert 'UNKNOWN' to NaN, flag invalid IDs
df['Vehicle_ID_Invalid'] = df['Vehicle_ID'].isna() | (df['Vehicle_ID'] == 'UNKNOWN')
df.loc[df['Vehicle_ID'] == 'UNKNOWN', 'Vehicle_ID'] = pd.NA

print("Cleaned Vehicle_IDs:")
print(sorted(df['Vehicle_ID'].dropna().unique()))
print(f"\n{df['Vehicle_ID'].value_counts().sort_index()}")

Cleaned Vehicle_IDs:
['CAR-001', 'CAR-002', 'CAR-003', 'CAR-005', 'CAR-006', 'CAR-007', 'CAR-008', 'CAR-009', 'CAR-010', 'CAR-011', 'CAR-012', 'CAR-013', 'CAR-014', 'CAR-015', 'CAR-016', 'CAR-017', 'CAR-018', 'CAR-019', 'CAR-020', 'CAR-021', 'CAR-022', 'CAR-023', 'CAR-024', 'CAR-025', 'CAR-026', 'CAR-027', 'CAR-028', 'CAR-029', 'CAR-030', 'CAR-031', 'CAR-032', 'CAR-033', 'CAR-034', 'CAR-035', 'CAR-036', 'CAR-037', 'CAR-038', 'CAR-039', 'CAR-04', 'CAR-040', 'CAR-041', 'CAR-042', 'CAR-043', 'CAR-044', 'CAR-045', 'CAR-046', 'CAR-047', 'CAR-048', 'CAR-049', 'CAR-050', 'CAR-051', 'CAR-052', 'CAR-053', 'CAR-054', 'CAR-055', 'CAR-056', 'CAR-057', 'CAR-058', 'CAR-059', 'CAR-060', 'CAR-061', 'CAR-062', 'CAR-063', 'CAR-064', 'CAR-065', 'CAR-066', 'CAR-067', 'CAR-068', 'CAR-069', 'CAR-070', 'CAR-071', 'CAR-072', 'CAR-073', 'CAR-074', 'CAR-075', 'CAR-076', 'CAR-077', 'CAR-078', 'CAR-079', 'CAR-080', 'CAR-081', 'CAR-082', 'CAR-083', 'CAR-084', 'CAR-085', 'CAR-086', 'CAR-087', 'CAR-088', 'CAR-089', 

In [ ]:
# Validate: all non-null IDs should match CAR-XX or CAR-XXX
valid_ids = df['Vehicle_ID'].dropna()
pattern_match = valid_ids.str.match(r'^CAR-\d{2,3}$')

print(f"Valid IDs: {len(valid_ids)} | Matching pattern: {pattern_match.sum()} | Non-matching: {(~pattern_match).sum()}")
if (~pattern_match).any():
    print(f"Non-matching: {valid_ids[~pattern_match].unique()}")

print(f"\nUnique Vehicle_IDs — Before: {unique_vid}, After: {df['Vehicle_ID'].nunique()}")
print(f"Invalid (NaN) Vehicle_IDs: {df['Vehicle_ID'].isna().sum()}")

Valid IDs: 4986 | Matching pattern: 4986 | Non-matching: 0

Unique Vehicle_IDs — Before: 601, After: 600
Invalid (NaN) Vehicle_IDs: 14


2. Timestamp normalization; invalid minutes fix; duration must be positive.

In [ ]:
import re
def fix_invalid_minutes(ts):
    if pd.isna(ts):
        return ts  
    match = re.search(r'(\d{1,2}):(\d{2})', str(ts))
    
    if match:
        hour = int(match.group(1))
        minute = int(match.group(2))
        
        if minute > 59:
            hour += minute // 60
            minute = minute % 60
        
        if hour > 23:
            hour = hour % 24
            
        ts = re.sub(r'\d{1,2}:\d{2}', f"{hour}:{minute:02d}", str(ts))
            
    return ts

In [ ]:
df["Pickup_TS"] = df["Pickup_TS"].apply(fix_invalid_minutes)
df["Return_TS"] = df["Return_TS"].apply(fix_invalid_minutes)
df["Booking_TS"] = df["Booking_TS"].apply(fix_invalid_minutes)
df.head(20)

,Reservation_ID,Customer_ID,Vehicle_ID,Vehicle_Class,Booking_Status,Booking_TS,Pickup_TS,Return_TS,Odo_Start,Odo_End,...,Promo_Code,City,GPS_Lat,GPS_Lon,Speed,Payment,Driver_License,Damage_Flag,Notes,Vehicle_ID_Invalid
0,RES-00001,CUST-1675,CAR-014,Hatchback,No_Show,02/13/2025 7:56,2025-02-18T7:56,18-02-2025 9:56,57815,"58,065",...,NaN,New Delhi,12.908959,77.604003,133kmh,cash,DL-2699413330,Major,Customer satisfied with ride,False
1,RES-00002,CUST-0853,CAR-230,SUV,Cancelled,08-02-2025 20:56,16-02-2025 20:56,2025-02-17 7:56,"77,762",77766km,...,NEW10,mumbai,12.946286,77.586710,55,CASH,DL-9806685378,Major,Customer requested early pickup,False
2,RES-00003,CUST-1181,CAR-001,Suzuki,Completed,2025-03-09T0:07,10-03-2025 0:07,12-03-2025 23:07,"57,856","58,238",...,DISC20,delhi,12.914102,77.627323,128 km/h,upi,DL-9542219877,Minor,Customer reported minor scratch on door,False
3,RES-00004,CUST-0182,CAR-434,Suzuki,Cancelled,04-01-2025 2:16,2025-01-11 2:16,11-01-2025 22:16,"59,967",60239km,...,SAVE50,delhi,12.873192,77.635268,25 km/h,cash,DL-4380657786,NaN,Customer requested early pickup,False
4,RES-00005,CUST-0356,CAR-121,Suzuki,No_Show,2025-05-04T4:22,2025-05-09T4:22,2025-05-10 11:22,"24,261","24,524",...,DISC20,bangalore,12.919640,77.598891,26kmh,-,DL-6429812852,Minor,AC performance slightly low,False
5,RES-00006,CUST-0061,CAR-360,Luxury,Cancelled,2025/5/27 9:18,2025-05-30T8:10,31-05-2025 9:10,"36,184","36,211",...,DISC20,chennai,12.872343,77.588487,100 km/h,card,DL-7859136756,Major,Customer reported minor scratch on door,False
6,RES-00007,CUST-0793,CAR-114,Creta,Completed,05-06-2025 14:21,08-06-2025 14:21,2025/6/9 8:02,"41,914",41881,...,WELCOME5,NaN,12.947693,77.603382,49,cash,DL-3024014652,Major,Tyre pressure alert during trip,False
7,RES-00008,CUST-0022,CAR-392,Luxury,No_Show,2025-03-28 2:32,2025/3/31 3:11,04/02/2025 5:32,"34,930",35217km,...,NEW10,Mumbai,12.927285,77.615525,fast,wallet,DL-2212993750,Major,AC performance slightly low,False
8,RES-00009,CUST-1700,CAR-505,Suzuki,No_Show,2025-04-08 12:30,2025-04-17T12:30,2025/04/17 11:30,66433,66519km,...,SAVE50,Bengaluru,12.922702,77.643999,135,netbanking,DL-1593968320,Minor,Low fuel warning observed,False
9,RES-00010,CUST-1586,CAR-448,EV,Cancelled,2025/3/15 22:18,2025/03/25 21:21,2025-03-26T5:21,"66,231",66471,...,DISC20,New Delhi,12.885247,77.596902,fast,netbanking,DL-9281443122,Major,Customer satisfied with ride,False


In [ ]:
df["Pickup_TS"] = df["Pickup_TS"].str.replace("/", "-", regex=False).str.replace("T", " ", regex=False)
df["Return_TS"] = df["Return_TS"].str.replace("/", "-", regex=False).str.replace("T", " ", regex=False)
df["Booking_TS"] = df["Booking_TS"].str.replace("/", "-", regex=False).str.replace("T", " ", regex=False)

In [ ]:
df["Pickup_TS"] = pd.to_datetime(df["Pickup_TS"], format='mixed', dayfirst=True)
df["Return_TS"] = pd.to_datetime(df["Return_TS"], format='mixed', dayfirst=True)
df["Booking_TS"] = pd.to_datetime(df["Booking_TS"], format='mixed', dayfirst=True)

In [ ]:
df.head(20)

,Reservation_ID,Customer_ID,Vehicle_ID,Vehicle_Class,Booking_Status,Booking_TS,Pickup_TS,Return_TS,Odo_Start,Odo_End,...,Promo_Code,City,GPS_Lat,GPS_Lon,Speed,Payment,Driver_License,Damage_Flag,Notes,Vehicle_ID_Invalid
0,RES-00001,CUST-1675,CAR-014,Hatchback,No_Show,2025-02-13 07:56:00,2025-02-18 07:56:00,2025-02-18 09:56:00,57815,"58,065",...,NaN,New Delhi,12.908959,77.604003,133kmh,cash,DL-2699413330,Major,Customer satisfied with ride,False
1,RES-00002,CUST-0853,CAR-230,SUV,Cancelled,2025-02-08 20:56:00,2025-02-16 20:56:00,2025-02-17 07:56:00,"77,762",77766km,...,NEW10,mumbai,12.946286,77.586710,55,CASH,DL-9806685378,Major,Customer requested early pickup,False
2,RES-00003,CUST-1181,CAR-001,Suzuki,Completed,2025-09-03 00:07:00,2025-03-10 00:07:00,2025-03-12 23:07:00,"57,856","58,238",...,DISC20,delhi,12.914102,77.627323,128 km/h,upi,DL-9542219877,Minor,Customer reported minor scratch on door,False
3,RES-00004,CUST-0182,CAR-434,Suzuki,Cancelled,2025-01-04 02:16:00,2025-11-01 02:16:00,2025-01-11 22:16:00,"59,967",60239km,...,SAVE50,delhi,12.873192,77.635268,25 km/h,cash,DL-4380657786,NaN,Customer requested early pickup,False
4,RES-00005,CUST-0356,CAR-121,Suzuki,No_Show,2025-04-05 04:22:00,2025-09-05 04:22:00,2025-10-05 11:22:00,"24,261","24,524",...,DISC20,bangalore,12.919640,77.598891,26kmh,-,DL-6429812852,Minor,AC performance slightly low,False
5,RES-00006,CUST-0061,CAR-360,Luxury,Cancelled,2025-05-27 09:18:00,2025-05-30 08:10:00,2025-05-31 09:10:00,"36,184","36,211",...,DISC20,chennai,12.872343,77.588487,100 km/h,card,DL-7859136756,Major,Customer reported minor scratch on door,False
6,RES-00007,CUST-0793,CAR-114,Creta,Completed,2025-06-05 14:21:00,2025-06-08 14:21:00,2025-09-06 08:02:00,"41,914",41881,...,WELCOME5,NaN,12.947693,77.603382,49,cash,DL-3024014652,Major,Tyre pressure alert during trip,False
7,RES-00008,CUST-0022,CAR-392,Luxury,No_Show,2025-03-28 02:32:00,2025-03-31 03:11:00,2025-02-04 05:32:00,"34,930",35217km,...,NEW10,Mumbai,12.927285,77.615525,fast,wallet,DL-2212993750,Major,AC performance slightly low,False
8,RES-00009,CUST-1700,CAR-505,Suzuki,No_Show,2025-08-04 12:30:00,2025-04-17 12:30:00,2025-04-17 11:30:00,66433,66519km,...,SAVE50,Bengaluru,12.922702,77.643999,135,netbanking,DL-1593968320,Minor,Low fuel warning observed,False
9,RES-00010,CUST-1586,CAR-448,EV,Cancelled,2025-03-15 22:18:00,2025-03-25 21:21:00,2025-03-26 05:21:00,"66,231",66471,...,DISC20,New Delhi,12.885247,77.596902,fast,netbanking,DL-9281443122,Major,Customer satisfied with ride,False


3. Odometer numeric extraction and unit unification (km)

In [ ]:
def clean_odometer(value):

    if pd.isna(value):
        return pd.NA

    # convert to string
    value = str(value)

    # remove commas (12,519 → 12519)
    value = value.replace(",", "")

    # remove non-numeric characters (remove km etc.)
    value = re.sub(r'[^0-9.]', '', value)

    if value == "":
        return pd.NA

    return float(value)
# Apply function to both odometer columns
df['Odo_Start'] = df['Odo_Start'].apply(clean_odometer)
df['Odo_End'] = df['Odo_End'].apply(clean_odometer)

print(df[['Odo_Start','Odo_End']].head(10))

   Odo_Start  Odo_End
0    57815.0  58065.0
1    77762.0  77766.0
2    57856.0  58238.0
3    59967.0  60239.0
4    24261.0  24524.0
5    36184.0  36211.0
6    41914.0  41881.0
7    34930.0  35217.0
8    66433.0  66519.0
9    66231.0  66471.0


4. Fuel level normalization (50%→0.5)

In [ ]:
# Remove % symbol
df["Fuel_Level"] = df["Fuel_Level"].astype(str).str.replace("%", "")

# Convert column to numeric
df["Fuel_Level"] = pd.to_numeric(df["Fuel_Level"], errors="coerce")

# Convert percentage values to fraction
df.loc[df["Fuel_Level"] > 1, "Fuel_Level"] = df["Fuel_Level"] / 100

df['Fuel_Level'].unique()

array([0.1 ,  nan, 0.3 , 0.5 , 0.75, 1.  , 0.25])

5. Rate parsing to numeric daily rate; currency normalization.

In [ ]:
usd_to_inr = 92
eur_to_inr = 106

def clean_rate(val):

    if pd.isna(val):
        return pd.NA

    val = str(val).replace(",", "").strip()

    # USD values
    if "USD" in val or "$" in val:
        num = re.findall(r'\d+', val)
        if num:
            return int(num[0]) * usd_to_inr

    # EURO values
    if "EUR" in val or "€" in val:
        num = re.findall(r'\d+', val)
        if num:
            return int(num[0]) * eur_to_inr

    # INR values
    num = re.findall(r'\d+', val)
    if num:
        return int(num[0])

    return pd.NA


df["Rate"] = df["Rate"].apply(clean_rate)

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
df.head(20)

,Reservation_ID,Customer_ID,Vehicle_ID,Vehicle_Class,Booking_Status,Booking_TS,Pickup_TS,Return_TS,Odo_Start,Odo_End,Fuel_Level,Rate,Promo_Code,City,GPS_Lat,GPS_Lon,Speed,Payment,Driver_License,Damage_Flag,Notes,Vehicle_ID_Invalid
0,RES-00001,CUST-1675,CAR-014,Hatchback,No_Show,2025-02-13 07:56:00,2025-02-18 07:56:00,2025-02-18 09:56:00,57815.0,58065.0,0.10,1840,NaN,New Delhi,12.908959,77.604003,133kmh,cash,DL-2699413330,Major,Customer satisfied with ride,False
1,RES-00002,CUST-0853,CAR-230,SUV,Cancelled,2025-02-08 20:56:00,2025-02-16 20:56:00,2025-02-17 07:56:00,77762.0,77766.0,NaN,2000,NEW10,mumbai,12.946286,77.586710,55,CASH,DL-9806685378,Major,Customer requested early pickup,False
2,RES-00003,CUST-1181,CAR-001,Suzuki,Completed,2025-09-03 00:07:00,2025-03-10 00:07:00,2025-03-12 23:07:00,57856.0,58238.0,0.30,2000,DISC20,delhi,12.914102,77.627323,128 km/h,upi,DL-9542219877,Minor,Customer reported minor scratch on door,False
3,RES-00004,CUST-0182,CAR-434,Suzuki,Cancelled,2025-01-04 02:16:00,2025-11-01 02:16:00,2025-01-11 22:16:00,59967.0,60239.0,0.50,1500,SAVE50,delhi,12.873192,77.635268,25 km/h,cash,DL-4380657786,NaN,Customer requested early pickup,False
4,RES-00005,CUST-0356,CAR-121,Suzuki,No_Show,2025-04-05 04:22:00,2025-09-05 04:22:00,2025-10-05 11:22:00,24261.0,24524.0,0.75,2300,DISC20,bangalore,12.919640,77.598891,26kmh,-,DL-6429812852,Minor,AC performance slightly low,False
5,RES-00006,CUST-0061,CAR-360,Luxury,Cancelled,2025-05-27 09:18:00,2025-05-30 08:10:00,2025-05-31 09:10:00,36184.0,36211.0,0.75,2500,DISC20,chennai,12.872343,77.588487,100 km/h,card,DL-7859136756,Major,Customer reported minor scratch on door,False
6,RES-00007,CUST-0793,CAR-114,Creta,Completed,2025-06-05 14:21:00,2025-06-08 14:21:00,2025-09-06 08:02:00,41914.0,41881.0,0.30,2000,WELCOME5,NaN,12.947693,77.603382,49,cash,DL-3024014652,Major,Tyre pressure alert during trip,False
7,RES-00008,CUST-0022,CAR-392,Luxury,No_Show,2025-03-28 02:32:00,2025-03-31 03:11:00,2025-02-04 05:32:00,34930.0,35217.0,0.75,1800,NEW10,Mumbai,12.927285,77.615525,fast,wallet,DL-2212993750,Major,AC performance slightly low,False
8,RES-00009,CUST-1700,CAR-505,Suzuki,No_Show,2025-08-04 12:30:00,2025-04-17 12:30:00,2025-04-17 11:30:00,66433.0,66519.0,1.00,1800,SAVE50,Bengaluru,12.922702,77.643999,135,netbanking,DL-1593968320,Minor,Low fuel warning observed,False
9,RES-00010,CUST-1586,CAR-448,EV,Cancelled,2025-03-15 22:18:00,2025-03-25 21:21:00,2025-03-26 05:21:00,66231.0,66471.0,0.25,<NA>,DISC20,New Delhi,12.885247,77.596902,fast,netbanking,DL-9281443122,Major,Customer satisfied with ride,False


6. City normalization to canonical names.

In [ ]:
# Check unique city values before cleaning
print("Cities BEFORE cleaning:")
print(df["City"].unique())

# Step 1: Remove spaces
df["City"] = df["City"].str.strip()

# Step 2: Convert everything to lowercase
df["City"] = df["City"].str.lower()

# Step 3: Replace unknown and missing values with NA
df["City"] = df["City"].replace("unknown", pd.NA)

# Step 4: Map inconsistent names to standard names
city_mapping = {
    "blr": "bengaluru",
    "bangalore": "bengaluru",
    "bengaluru": "bengaluru",

    "mum": "mumbai",
    "bombay": "mumbai",
    "mumbai": "mumbai",

    "del": "delhi",
    "delhi": "delhi",
    "new delhi": "delhi",

    "chn": "chennai",
    "chennai": "chennai",

}

df["City"] = df["City"].replace(city_mapping)

# Step 5: Convert to title case
df["City"] = df["City"].str.title()

df["City"].value_counts()

# Check unique city values after cleaning
print("Cities AFTER cleaning:")
print(df["City"].unique())

Cities BEFORE cleaning:
<StringArray>
['New Delhi',    'mumbai',     'delhi', 'bangalore',   'chennai',         nan,
    'Mumbai', 'Bengaluru',       'MUM',   'Chennai',       'BLR',     'Delhi',
       'CHN',   'unknown',    'Bombay',       'blr',     'DELHI']
Length: 17, dtype: str
Cities AFTER cleaning:
<StringArray>
['Delhi', 'Mumbai', 'Bengaluru', 'Chennai', nan]
Length: 5, dtype: str


7. Duplicate reservation dedup (same ID).

In [ ]:
print(f"Duplicate Reservation_IDs: {df['Reservation_ID'].duplicated().sum()}")
df = df.drop_duplicates(subset='Reservation_ID', keep='first')
print(f"Shape after dedup: {df.shape}")

Duplicate Reservation_IDs: 473
Shape after dedup: (4527, 22)


8. Return before pickup rule validation.

In [ ]:
swap_mask = df["Return_TS"] < df["Pickup_TS"]

df.loc[swap_mask, ["Pickup_TS","Return_TS"]] = \
df.loc[swap_mask, ["Return_TS","Pickup_TS"]].values

In [ ]:
df["Duration_Hours"] = (df["Return_TS"] - df["Pickup_TS"]).dt.total_seconds() / 3600

In [ ]:
invalid_duration = df[df["Duration_Hours"] < 0]

In [ ]:
equal_mask = df["Return_TS"] == df["Pickup_TS"]

In [ ]:
df.loc[equal_mask, ["Pickup_TS", "Return_TS"]] = pd.NaT

In [ ]:
print(invalid_duration)

Empty DataFrame
Columns: [Reservation_ID, Customer_ID, Vehicle_ID, Vehicle_Class, Booking_Status, Booking_TS, Pickup_TS, Return_TS, Odo_Start, Odo_End, Fuel_Level, Rate, Promo_Code, City, GPS_Lat, GPS_Lon, Speed, Payment, Driver_License, Damage_Flag, Notes, Vehicle_ID_Invalid, Duration_Hours]
Index: []


In [ ]:
df.head(20)

,Reservation_ID,Customer_ID,Vehicle_ID,Vehicle_Class,Booking_Status,Booking_TS,Pickup_TS,Return_TS,Odo_Start,Odo_End,Fuel_Level,Rate,Promo_Code,City,GPS_Lat,GPS_Lon,Speed,Payment,Driver_License,Damage_Flag,Notes,Vehicle_ID_Invalid,Duration_Hours
0,RES-00001,CUST-1675,CAR-014,Hatchback,No_Show,2025-02-13 07:56:00,2025-02-18 07:56:00,2025-02-18 09:56:00,57815.0,58065.0,0.10,1840,NaN,Delhi,12.908959,77.604003,133kmh,cash,DL-2699413330,Major,Customer satisfied with ride,False,2.000000
1,RES-00002,CUST-0853,CAR-230,SUV,Cancelled,2025-02-08 20:56:00,2025-02-16 20:56:00,2025-02-17 07:56:00,77762.0,77766.0,NaN,2000,NEW10,Mumbai,12.946286,77.586710,55,CASH,DL-9806685378,Major,Customer requested early pickup,False,11.000000
2,RES-00003,CUST-1181,CAR-001,Suzuki,Completed,2025-09-03 00:07:00,2025-03-10 00:07:00,2025-03-12 23:07:00,57856.0,58238.0,0.30,2000,DISC20,Delhi,12.914102,77.627323,128 km/h,upi,DL-9542219877,Minor,Customer reported minor scratch on door,False,71.000000
3,RES-00004,CUST-0182,CAR-434,Suzuki,Cancelled,2025-01-04 02:16:00,2025-01-11 22:16:00,2025-11-01 02:16:00,59967.0,60239.0,0.50,1500,SAVE50,Delhi,12.873192,77.635268,25 km/h,cash,DL-4380657786,NaN,Customer requested early pickup,False,7036.000000
4,RES-00005,CUST-0356,CAR-121,Suzuki,No_Show,2025-04-05 04:22:00,2025-09-05 04:22:00,2025-10-05 11:22:00,24261.0,24524.0,0.75,2300,DISC20,Bengaluru,12.919640,77.598891,26kmh,-,DL-6429812852,Minor,AC performance slightly low,False,727.000000
5,RES-00006,CUST-0061,CAR-360,Luxury,Cancelled,2025-05-27 09:18:00,2025-05-30 08:10:00,2025-05-31 09:10:00,36184.0,36211.0,0.75,2500,DISC20,Chennai,12.872343,77.588487,100 km/h,card,DL-7859136756,Major,Customer reported minor scratch on door,False,25.000000
6,RES-00007,CUST-0793,CAR-114,Creta,Completed,2025-06-05 14:21:00,2025-06-08 14:21:00,2025-09-06 08:02:00,41914.0,41881.0,0.30,2000,WELCOME5,NaN,12.947693,77.603382,49,cash,DL-3024014652,Major,Tyre pressure alert during trip,False,2153.683333
7,RES-00008,CUST-0022,CAR-392,Luxury,No_Show,2025-03-28 02:32:00,2025-02-04 05:32:00,2025-03-31 03:11:00,34930.0,35217.0,0.75,1800,NEW10,Mumbai,12.927285,77.615525,fast,wallet,DL-2212993750,Major,AC performance slightly low,False,1317.650000
8,RES-00009,CUST-1700,CAR-505,Suzuki,No_Show,2025-08-04 12:30:00,2025-04-17 11:30:00,2025-04-17 12:30:00,66433.0,66519.0,1.00,1800,SAVE50,Bengaluru,12.922702,77.643999,135,netbanking,DL-1593968320,Minor,Low fuel warning observed,False,1.000000
9,RES-00010,CUST-1586,CAR-448,EV,Cancelled,2025-03-15 22:18:00,2025-03-25 21:21:00,2025-03-26 05:21:00,66231.0,66471.0,0.25,<NA>,DISC20,Delhi,12.885247,77.596902,fast,netbanking,DL-9281443122,Major,Customer satisfied with ride,False,8.000000


9. Payment method standardization (UPI/CARD/CASH/WALLET).

In [ ]:
df["Payment"] = df["Payment"].astype(str).str.lower().str.strip()

payment_dict = {
    "upi": "UPI",
    
    "credit card": "CARD",
    "debit card": "CARD",
    "card": "CARD",
    "netbanking": "CARD",
    
    "cash": "CASH",
    
    "wallet": "WALLET",
    
    "-": pd.NA,
    "nan": pd.NA
}

df["Payment"] = df["Payment"].replace(payment_dict)
print(df["Payment"].unique())

<StringArray>
['CASH', 'UPI', nan, 'CARD', 'WALLET']
Length: 5, dtype: str


10. Mileage sanity check (End ≥ Start)

In [ ]:
mileage_check = df[['Reservation_ID','Odo_Start','Odo_End']].copy()

# Swap Odo_Start and Odo_End where End < Start (likely data entry error)
odo_swap_mask = df['Odo_End'] < df['Odo_Start']
df.loc[odo_swap_mask, ['Odo_Start', 'Odo_End']] = df.loc[odo_swap_mask, ['Odo_End', 'Odo_Start']].values
print(f"Swapped Odo_Start/Odo_End for {odo_swap_mask.sum()} rows")

# Verify no invalid mileages remain
mileage_check = df[['Reservation_ID','Odo_Start','Odo_End']].copy()
mileage_check['Mileage_Valid'] = mileage_check['Odo_End'] >= mileage_check['Odo_Start']
print(f"Invalid mileages remaining: {(~mileage_check['Mileage_Valid']).sum()}")
mileage_check.head(10)

Swapped Odo_Start/Odo_End for 745 rows
Invalid mileages remaining: 0


,Reservation_ID,Odo_Start,Odo_End,Mileage_Valid
0,RES-00001,57815.0,58065.0,True
1,RES-00002,77762.0,77766.0,True
2,RES-00003,57856.0,58238.0,True
3,RES-00004,59967.0,60239.0,True
4,RES-00005,24261.0,24524.0,True
5,RES-00006,36184.0,36211.0,True
6,RES-00007,41881.0,41914.0,True
7,RES-00008,34930.0,35217.0,True
8,RES-00009,66433.0,66519.0,True
9,RES-00010,66231.0,66471.0,True


11. Refueling event detection vs fuel change and distance.

In [ ]:
# Step 1: Normalize Fuel_Level (re-normalize in case of any remaining % values)
df["Fuel_Level"] = df["Fuel_Level"].astype(str).str.replace("%", "", regex=False)
df["Fuel_Level"] = pd.to_numeric(df["Fuel_Level"], errors="coerce")

# Convert percentage values to fraction
df.loc[df["Fuel_Level"] > 1, "Fuel_Level"] = df["Fuel_Level"] / 100

# Step 2: Prepare validity mask and distance driven directly on main dataframe
valid_odo_mask = df["Odo_End"] >= df["Odo_Start"]
df["Distance_Driven"] = pd.NA
df.loc[valid_odo_mask, "Distance_Driven"] = (
    df.loc[valid_odo_mask, "Odo_End"] - df.loc[valid_odo_mask, "Odo_Start"]
)

# Step 3: Detect refuel events directly on main dataframe
sorted_idx = df.sort_values(["Vehicle_ID", "Odo_Start"]).index
fuel_diff = df.loc[sorted_idx].groupby("Vehicle_ID")["Fuel_Level"].diff()
refuel_flag = (fuel_diff > 0).map({True: "Refueled", False: "No Refuel"})

df["Refuel_Event"] = pd.NA
df.loc[refuel_flag.index, "Refuel_Event"] = refuel_flag
df.loc[~valid_odo_mask, "Refuel_Event"] = pd.NA

# Step 4: Display
df.head(10)
df['Refuel_Event'].unique()

array(['No Refuel', 'Refueled'], dtype=object)

12. Vehicle availability overlap checks.

In [ ]:
df = df.sort_values(['Vehicle_ID','Pickup_TS'])

df['Prev_Return'] = df.groupby('Vehicle_ID')['Return_TS'].shift(1)

df['Overlap_Flag'] = df['Pickup_TS'] < df['Prev_Return']

df['Overlap_Flag'].value_counts()

Overlap_Flag
False    3235
True     1292
Name: count, dtype: int64

13. Damage / Incident Log Linkage to Reservation

In [ ]:
#inspect
df[['Reservation_ID','Vehicle_ID','Customer_ID','Damage_Flag','Notes']].head()

#Extract rows where damage was reported
damage_logs = df[
    df['Damage_Flag'].notna() &
    (df['Damage_Flag'] != 'None')
]


#Create a separate incident log dataset
incident_log = damage_logs[[
    'Reservation_ID',
    'Vehicle_ID',
    'Customer_ID',
    'Damage_Flag',
    'Notes'
]].copy()


#Check for records where damage is flagged but notes are missing
missing_notes = incident_log[
    incident_log['Notes'].isna() |
    (incident_log['Notes'].str.strip() == "")
]


print("Damage incidents missing description:", len(missing_notes))


#Detect possible damage mentioned in notes but flag is 'None'
possible_damage = df[
    (df['Damage_Flag'] == 'None') &
    (df['Notes'].str.contains('damage|scratch|dent|crack', case=False, na=False))
]


print("Possible misclassified incidents:", len(possible_damage))


#Display the final incident log dataset
incident_log.head(15)
df['Notes'].unique()


Damage incidents missing description: 500
Possible misclassified incidents: 0


<StringArray>
['Customer reported minor scratch on door',
          'Car returned in good condition',
               'Low fuel warning observed',
         'Tyre pressure alert during trip',
             'AC performance slightly low',
                                       nan,
    'Vehicle returned late due to traffic',
         'Customer requested early pickup',
              'Interior cleaning required',
  'Navigation system malfunction reported',
            'Customer satisfied with ride']
Length: 11, dtype: str

14. Driver license masking and validation (if present).

In [ ]:
#validation
def validate_license(x):
    if pd.isna(x):
        return None
    
    x = str(x)
    
    if re.match(r"DL-\d{10}$", x):
        return x
    else:
        return None

df["Driver_License"] = df["Driver_License"].apply(validate_license)


In [ ]:
#masking
def mask_license(x):
    if pd.isna(x):
        return None
    
    return x[:7] + "XXXXXX"

df["Driver_License"] = df["Driver_License"].apply(mask_license)

15. Promo/coupon code validation & expiry checks.

In [ ]:
# convert pickup timestamp
df["Pickup_TS"] = pd.to_datetime(df["Pickup_TS"], errors="coerce")

# promo expiry dictionary
promo_expiry = {
    "NEW10": "2026-03-31",
    "DISC20": "2026-02-28",
    "SAVE50": "2026-06-30",
    "WELCOME5": "2026-01-31"
}

# convert expiry values to datetime
promo_expiry = {k: pd.to_datetime(v) for k, v in promo_expiry.items()}

# map expiry date to dataset
df["Promo_Expiry"] = df["Promo_Code"].map(promo_expiry)

df["Promo_Status"] = "Valid"

# invalid promo codes
df.loc[df["Promo_Code"].notna() & df["Promo_Expiry"].isna(), "Promo_Status"] = "Invalid_Code"

# expired promo codes
df.loc[(df["Promo_Expiry"].notna()) & (df["Pickup_TS"] > df["Promo_Expiry"]), "Promo_Status"] = "Expired"



In [ ]:
df.head(30)


,Reservation_ID,Customer_ID,Vehicle_ID,Vehicle_Class,Booking_Status,Booking_TS,Pickup_TS,Return_TS,Odo_Start,Odo_End,Fuel_Level,Rate,Promo_Code,City,GPS_Lat,GPS_Lon,Speed,Payment,Driver_License,Damage_Flag,Notes,Vehicle_ID_Invalid,Duration_Hours,Distance_Driven,Refuel_Event,Prev_Return,Overlap_Flag,Promo_Expiry,Promo_Status
2,RES-00003,CUST-1181,CAR-001,Suzuki,Completed,2025-09-03 00:07:00,2025-03-10 00:07:00,2025-03-12 23:07:00,57856.0,58238.0,0.30,2000,DISC20,Delhi,12.914102,77.627323,128 km/h,UPI,DL-9542XXXXXX,Minor,Customer reported minor scratch on door,False,71.000000,382.0,Refueled,NaT,False,2026-02-28,Valid
3654,RES-03655,CUST-0243,CAR-001,Suzuki,Completed,2025-08-04 06:48:00,2025-04-13 06:48:00,2025-04-14 10:48:00,79962.0,80173.0,0.30,2300,WELCOME5,Mumbai,12.886153,77.550280,49kmh,WALLET,DL-9194XXXXXX,Major,Car returned in good condition,False,28.000000,211.0,No Refuel,2025-03-12 23:07:00,False,2026-01-31,Valid
1773,RES-01774,CUST-1755,CAR-001,Suzuki,Cancelled,2025-04-13 00:12:00,2025-04-15 00:12:00,2025-04-15 03:12:00,57390.0,57599.0,0.10,1840,DISC20,Mumbai,12.886021,77.567646,84 km/h,CARD,DL-8187XXXXXX,Minor,Car returned in good condition,False,3.000000,209.0,No Refuel,2025-04-14 10:48:00,False,2026-02-28,Valid
3808,RES-03809,CUST-0799,CAR-001,Suzuki,Cancelled,2025-06-05 00:50:00,2025-06-14 00:50:00,2025-06-15 14:50:00,37275.0,37463.0,0.50,2000,NEW10,Chennai,12.924378,77.625311,76kmh,CARD,DL-3438XXXXXX,Minor,Low fuel warning observed,False,38.000000,188.0,No Refuel,2025-04-15 03:12:00,False,2026-03-31,Valid
1292,RES-01293,CUST-0799,CAR-001,Suzuki,Cancelled,2025-05-01 21:46:00,2025-09-01 21:46:00,2025-12-01 00:46:00,70972.0,70992.0,0.75,1500,DISC20,Mumbai,12.899210,77.624153,NaN,CARD,DL-3438XXXXXX,Minor,Tyre pressure alert during trip,False,2163.000000,20.0,Refueled,2025-06-15 14:50:00,False,2026-02-28,Valid
2592,RES-02593,CUST-1437,CAR-001,Suzuki,Completed,2025-07-06 20:51:00,2025-09-06 20:51:00,2025-10-06 07:51:00,60442.0,60884.0,0.30,2300,DISC20,Delhi,12.948511,77.637527,fast,CARD,DL-9254XXXXXX,NaN,AC performance slightly low,False,707.000000,442.0,No Refuel,2025-12-01 00:46:00,True,2026-02-28,Valid
1460,RES-01461,CUST-0555,CAR-002,Toyota,Completed,2024-12-23 20:04:00,2025-01-01 20:04:00,2025-01-02 20:04:00,39548.0,39636.0,0.10,<NA>,NaN,Delhi,12.914726,77.612034,28kmh,UPI,DL-8144XXXXXX,Minor,AC performance slightly low,False,24.000000,88.0,No Refuel,NaT,False,NaT,Valid
2264,RES-02265,CUST-0117,CAR-002,Toyota,Cancelled,2025-03-05 01:10:00,2025-03-14 01:10:00,2025-11-03 01:10:00,13151.0,13215.0,0.50,2000,NEW10,Mumbai,12.933910,77.563362,138kmh,UPI,DL-5814XXXXXX,Minor,NaN,False,5616.000000,64.0,No Refuel,2025-01-02 20:04:00,False,2026-03-31,Valid
3308,RES-03309,CUST-0243,CAR-002,Toyota,Completed,2025-11-03 21:06:00,2025-03-17 21:06:00,2025-03-19 06:06:00,41606.0,41887.0,0.30,1500,DISC20,Mumbai,12.877574,77.567612,53,NaN,DL-9194XXXXXX,Major,Low fuel warning observed,False,33.000000,281.0,Refueled,2025-11-03 01:10:00,True,2026-02-28,Valid
2193,RES-02194,CUST-1667,CAR-002,Toyota,No_Show,2025-05-25 20:25:00,2025-04-06 15:25:00,2025-06-03 20:25:00,29008.0,29112.0,0.10,1500,NaN,Chennai,12.874858,77.646015,46kmh,CARD,DL-2386XXXXXX,NaN,NaN,False,1397.000000,104.0,No Refuel,2025-03-19 06:06:00,False,NaT,Valid


16. Telematics GPS join and jitter smoothing.

In [ ]:
# 1 check duplicates
df[['GPS_Lat', 'GPS_Lon']].duplicated().sum()

# 2 clean quote artifacts, convert to numeric, and handle missing values
df['GPS_Lat'] = pd.to_numeric(
    df['GPS_Lat'].astype(str).str.strip().str.replace(r"^'+|'+$", "", regex=True),
    errors='coerce'
 )
df['GPS_Lon'] = pd.to_numeric(
    df['GPS_Lon'].astype(str).str.strip().str.replace(r"^'+|'+$", "", regex=True),
    errors='coerce'
 )
df['GPS_Lat'] = df['GPS_Lat'].interpolate()
df['GPS_Lon'] = df['GPS_Lon'].interpolate()

# 3 smooth jitter and round to 4 decimals
df['GPS_Lat'] = df['GPS_Lat'].rolling(window=3, min_periods=1).mean().round(4)
df['GPS_Lon'] = df['GPS_Lon'].rolling(window=3, min_periods=1).mean().round(4)

df[['GPS_Lat', 'GPS_Lon']].dtypes

GPS_Lat    float64
GPS_Lon    float64
dtype: object

17. Speeding/harsh events normalization from telematics.

In [ ]:
# Inspect Speed Column
df["Speed"].head()

# Convert Speed Column to String
df["Speed"] = df["Speed"].astype(str)

# Remove Units (kmh and km/h)
df["Speed"] = df["Speed"].str.replace("km/h","", regex=False)
df["Speed"] = df["Speed"].str.replace("kmh","", regex=False)

# Convert "fast" to Numeric Speed

# Since "fast" means driving faster than normal, we convert it to 90 km/h.

df["Speed"] = df["Speed"].replace("fast", 90)

# Convert Entire Column to Numeric
df["Speed"] = pd.to_numeric(df["Speed"], errors="coerce")


# Create Driver Behavior Classification

def classify_speed(speed):

    if pd.isna(speed):
        return "Unknown"

    elif speed <= 80:
        return "Normal Driving"

    elif speed <= 100:
        return "Fast Driving"

    else:
        return "Speeding"


df["Driver_Behavior"] = df["Speed"].apply(classify_speed)


# Verify Result

df[["Speed","Driver_Behavior"]].head(10)

,Speed,Driver_Behavior
2,128.0,Speeding
3654,49.0,Normal Driving
1773,84.0,Fast Driving
3808,76.0,Normal Driving
1292,NaN,Unknown
2592,90.0,Fast Driving
1460,28.0,Normal Driving
2264,138.0,Speeding
3308,53.0,Normal Driving
2193,46.0,Normal Driving


18. PII redaction in notes and feedback.

In [ ]:
df['Notes'] = df['Notes'].str.replace(r'\b\d{10}\b', '[REDACTED]', regex=True)
df['Notes'] = df['Notes'].str.replace(r'\S+@\S+', '[REDACTED]', regex=True)
df['Notes'] = df['Notes'].str.replace(r'\b[A-Z0-9]{8,}\b', '[REDACTED]', regex=True)

19. Rate plan mapping to master tariffs.

In [ ]:
df['Rate'] = df['Rate'].astype(str)
df['Rate'] = df['Rate'].str.replace('[^0-9.]', '', regex=True)
df['Rate'] = pd.to_numeric(df['Rate'], errors='coerce')

20. Tax/GST computation validation.

In [ ]:
# USE CASE 20: Tax/GST computation validation (hour-based billing)

df["Rate"] = pd.to_numeric(df["Rate"], errors="coerce")
df["Duration_Hours"] = pd.to_numeric(df["Duration_Hours"], errors="coerce")

billable_hours = df["Duration_Hours"].clip(lower=0).fillna(0)
df["Total_Amount"] = ((df["Rate"] / 24) * billable_hours * 1.18).round(2)

# Only completed trips are billable in this synthetic pipeline
df.loc[~df["Booking_Status"].eq("Completed"), "Total_Amount"] = 0.0

In [ ]:
df['Vehicle_ID'].unique()

<StringArray>
['CAR-001', 'CAR-002', 'CAR-003', 'CAR-005', 'CAR-006', 'CAR-007', 'CAR-008',
 'CAR-009', 'CAR-010', 'CAR-011',
 ...
 'CAR-592', 'CAR-593', 'CAR-594', 'CAR-595', 'CAR-596', 'CAR-597', 'CAR-598',
 'CAR-599', 'CAR-600',       nan]
Length: 601, dtype: str

In [ ]:
df["Pickup_TS"] = pd.to_datetime(df["Pickup_TS"])
df["Return_TS"] = pd.to_datetime(df["Return_TS"])

df["Pickup_TS"] = df["Pickup_TS"].fillna(df["Pickup_TS"].mean())
df["Return_TS"] = df["Return_TS"].fillna(df["Return_TS"].mean())
df["Pickup_TS"] = df["Pickup_TS"].fillna(df["Pickup_TS"].mean())
df["Return_TS"] = df["Return_TS"].fillna(df["Return_TS"].mean())

df["Fuel_Level"] = df["Fuel_Level"].fillna(df["Fuel_Level"].mean())
df["Rate"] = df["Rate"].fillna(df["Rate"].mean())


df["Promo_Code"] = df["Promo_Code"].fillna(df["Promo_Code"].mode()[0])
df["City"] = df["City"].fillna(df["City"].mode()[0])


df["Speed"] = df["Speed"].fillna(df["Speed"].median())
df["Payment"] = df["Payment"].fillna(df["Payment"].mode()[0])

df["Damage_Flag"] = df["Damage_Flag"].fillna("None")

df["Notes"] = df["Notes"].fillna("No Notes")

df["Duration_Hours"] = df["Duration_Hours"].round(2)

In [ ]:
df["Booking_TS"] = pd.to_datetime(df["Booking_TS"])
df["Pickup_TS"] = pd.to_datetime(df["Pickup_TS"])

df.loc[df["Booking_TS"] > df["Pickup_TS"], "Booking_TS"] = df["Pickup_TS"] - pd.Timedelta(days=1)
df.head(20)

,Reservation_ID,Customer_ID,Vehicle_ID,Vehicle_Class,Booking_Status,Booking_TS,Pickup_TS,Return_TS,Odo_Start,Odo_End,Fuel_Level,Rate,Promo_Code,City,GPS_Lat,GPS_Lon,Speed,Payment,Driver_License,Damage_Flag,Notes,Vehicle_ID_Invalid,Duration_Hours,Distance_Driven,Refuel_Event,Prev_Return,Overlap_Flag,Promo_Expiry,Promo_Status,Driver_Behavior,Total_Amount
2,RES-00003,CUST-1181,CAR-001,Suzuki,Completed,2025-03-09 00:07:00,2025-03-10 00:07:00,2025-03-12 23:07:00,57856.0,58238.0,0.300000,2000.000000,DISC20,Delhi,12.9141,77.6273,128.0,UPI,DL-9542XXXXXX,Minor,Customer reported minor scratch on door,False,71.00,382.0,Refueled,NaT,False,2026-02-28,Valid,Speeding,2360.0
3654,RES-03655,CUST-0243,CAR-001,Suzuki,Completed,2025-04-12 06:48:00,2025-04-13 06:48:00,2025-04-14 10:48:00,79962.0,80173.0,0.300000,2300.000000,WELCOME5,Mumbai,12.9001,77.5888,49.0,WALLET,DL-9194XXXXXX,Major,Car returned in good condition,False,28.00,211.0,No Refuel,2025-03-12 23:07:00,False,2026-01-31,Valid,Normal Driving,2714.0
1773,RES-01774,CUST-1755,CAR-001,Suzuki,Cancelled,2025-04-13 00:12:00,2025-04-15 00:12:00,2025-04-15 03:12:00,57390.0,57599.0,0.100000,1840.000000,DISC20,Mumbai,12.8954,77.5817,84.0,CARD,DL-8187XXXXXX,Minor,Car returned in good condition,False,3.00,209.0,No Refuel,2025-04-14 10:48:00,False,2026-02-28,Valid,Fast Driving,2171.2
3808,RES-03809,CUST-0799,CAR-001,Suzuki,Cancelled,2025-06-05 00:50:00,2025-06-14 00:50:00,2025-06-15 14:50:00,37275.0,37463.0,0.500000,2000.000000,NEW10,Chennai,12.8989,77.5811,76.0,CARD,DL-3438XXXXXX,Minor,Low fuel warning observed,False,38.00,188.0,No Refuel,2025-04-15 03:12:00,False,2026-03-31,Valid,Normal Driving,2360.0
1292,RES-01293,CUST-0799,CAR-001,Suzuki,Cancelled,2025-05-01 21:46:00,2025-09-01 21:46:00,2025-12-01 00:46:00,70972.0,70992.0,0.750000,1500.000000,DISC20,Mumbai,12.9032,77.6057,90.0,CARD,DL-3438XXXXXX,Minor,Tyre pressure alert during trip,False,2163.00,20.0,Refueled,2025-06-15 14:50:00,False,2026-02-28,Valid,Unknown,1770.0
2592,RES-02593,CUST-1437,CAR-001,Suzuki,Completed,2025-07-06 20:51:00,2025-09-06 20:51:00,2025-10-06 07:51:00,60442.0,60884.0,0.300000,2300.000000,DISC20,Delhi,12.9240,77.6290,90.0,CARD,DL-9254XXXXXX,None,AC performance slightly low,False,707.00,442.0,No Refuel,2025-12-01 00:46:00,True,2026-02-28,Valid,Fast Driving,2714.0
1460,RES-01461,CUST-0555,CAR-002,Toyota,Completed,2024-12-23 20:04:00,2025-01-01 20:04:00,2025-01-02 20:04:00,39548.0,39636.0,0.100000,1835.274591,NEW10,Delhi,12.9208,77.6246,28.0,UPI,DL-8144XXXXXX,Minor,AC performance slightly low,False,24.00,88.0,No Refuel,NaT,False,NaT,Valid,Normal Driving,NaN
2264,RES-02265,CUST-0117,CAR-002,Toyota,Cancelled,2025-03-05 01:10:00,2025-03-14 01:10:00,2025-11-03 01:10:00,13151.0,13215.0,0.500000,2000.000000,NEW10,Mumbai,12.9324,77.6043,138.0,UPI,DL-5814XXXXXX,Minor,No Notes,False,5616.00,64.0,No Refuel,2025-01-02 20:04:00,False,2026-03-31,Valid,Speeding,2360.0
3308,RES-03309,CUST-0243,CAR-002,Toyota,Completed,2025-03-16 21:06:00,2025-03-17 21:06:00,2025-03-19 06:06:00,41606.0,41887.0,0.300000,1500.000000,DISC20,Mumbai,12.9087,77.5810,53.0,CARD,DL-9194XXXXXX,Major,Low fuel warning observed,False,33.00,281.0,Refueled,2025-11-03 01:10:00,True,2026-02-28,Valid,Normal Driving,1770.0
2193,RES-02194,CUST-1667,CAR-002,Toyota,No_Show,2025-04-05 15:25:00,2025-04-06 15:25:00,2025-06-03 20:25:00,29008.0,29112.0,0.100000,1500.000000,NEW10,Chennai,12.8954,77.5923,46.0,CARD,DL-2386XXXXXX,None,No Notes,False,1397.00,104.0,No Refuel,2025-03-19 06:06:00,False,NaT,Valid,Normal Driving,1770.0


In [ ]:
df.loc[df["Booking_Status"] == "Cancelled", "Odo_End"] = df["Odo_Start"]
df.loc[df["Booking_Status"] == "Cancelled", "Total_Amount"] = 0

df["Total_Amount"] = df["Total_Amount"].fillna(df["Total_Amount"].mean())

In [ ]:
df["Fuel_Level"] = df["Fuel_Level"].round(2)
df["Rate"] = df["Rate"].round(2)
df["Total_Amount"] = df["Total_Amount"].round(2)

In [ ]:
df["Distance_Driven"] = df["Odo_End"] - df["Odo_Start"]
df["Vehicle_ID"] = df["Vehicle_ID"].fillna("CAR-600")
df.loc[df["Promo_Expiry"].isnull(), "Promo_Status"] = "Invalid"

In [ ]:
import numpy as np
from geopy.geocoders import Nominatim
import time

# Initialize geocoder
geolocator = Nominatim(user_agent="city_lat_long_generator")

# Get unique cities
cities = df["City"].dropna().unique()

city_coords = {}

# Get base coordinates for each city
for city in cities:
    try:
        location = geolocator.geocode(city)
        if location:
            city_coords[city] = (location.latitude, location.longitude)
        else:
            city_coords[city] = (np.nan, np.nan)
        time.sleep(1)
    except:
        city_coords[city] = (np.nan, np.nan)

# Function to generate random nearby coordinates
def generate_random_coords(city):
    lat, lon = city_coords.get(city, (np.nan, np.nan))

    if pd.isna(lat) or pd.isna(lon):
        return pd.Series([np.nan, np.nan])

    lat_random = lat + np.random.uniform(-0.02, 0.02)
    lon_random = lon + np.random.uniform(-0.02, 0.02)

    # round to 4 decimal places
    lat_random = round(lat_random, 4)
    lon_random = round(lon_random, 4)

    return pd.Series([lat_random, lon_random])

# Apply to dataframe
df[["GPS_Lat", "GPS_Lon"]] = df["City"].apply(generate_random_coords)

In [ ]:
# Save cleaned dataset to CSV
df.to_csv("../Datasets/car_rental_cleaned_dataset.csv", index=False)
print("Cleaned dataset saved to 'car_rental_cleaned_dataset.csv'")
print(f"Final dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

Cleaned dataset saved to 'car_rental_cleaned_dataset.csv'
Final dataset shape: (4527, 31)
Columns: ['Reservation_ID', 'Customer_ID', 'Vehicle_ID', 'Vehicle_Class', 'Booking_Status', 'Booking_TS', 'Pickup_TS', 'Return_TS', 'Odo_Start', 'Odo_End', 'Fuel_Level', 'Rate', 'Promo_Code', 'City', 'GPS_Lat', 'GPS_Lon', 'Speed', 'Payment', 'Driver_License', 'Damage_Flag', 'Notes', 'Vehicle_ID_Invalid', 'Duration_Hours', 'Distance_Driven', 'Refuel_Event', 'Prev_Return', 'Overlap_Flag', 'Promo_Expiry', 'Promo_Status', 'Driver_Behavior', 'Total_Amount']
